# Case Management Trend Dashboard

**Purpose**: Analyze historical case management performance, identify trends in resolution times and case outcomes, and evaluate investigator productivity and quality over time.

**Dashboard Features**:
- Flexible date range selection with pre-configured periods
- Multiple time granularities (Daily, Weekly, Monthly, Quarterly)
- Trend indicators comparing current period to previous period
- Supervisory-level insights without sensitive transaction details
- Gold-layer data for optimized performance
- HTML and JSON export for reporting and scheduled delivery

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf

spark_jars = os.environ.get("SPARK_JARS", "")
jar_list = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "http://localhost:9878")
s3a_access_key = os.environ.get("S3A_ACCESS_KEY", "tazama")
s3a_secret_key = os.environ.get("S3A_SECRET_KEY", "tazama")
driver_memory = os.environ.get("SPARK_DRIVER_MEMORY", "4g")

spark = (
    SparkSession.builder
    .appName("Tazama-Dashboard")
    .master("local[*]")
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", s3a_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl.disable.cache", "true")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory", driver_memory)
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128mb")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import json
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")
alerts_path = f"{WAREHOUSE_ROOT}/gold/alerts"
cases_path = f"{WAREHOUSE_ROOT}/gold/cases"
tasks_path = f"{WAREHOUSE_ROOT}/gold/tasks"
transactions_path = f"{WAREHOUSE_ROOT}/gold/transactions"

# Load Gold-layer data
print("\n" + "="*80)
print("LOADING GOLD-LAYER DATA")
print("="*80)

alerts = spark.read.format("hudi").load(alerts_path)
cases = spark.read.format("hudi").load(cases_path)
tasks = spark.read.format("hudi").load(tasks_path)
transactions = spark.read.format("hudi").load(transactions_path)

print(f"✓ Alerts loaded: {alerts.count()} records")
print(f"✓ Cases loaded: {cases.count()} records")
print(f"✓ Tasks loaded: {tasks.count()} records")
print(f"✓ Transactions loaded: {transactions.count()} records")
print("="*80)

26/04/17 06:58:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 06:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/17 06:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/17 06:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/17 06:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/04/17 06:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/04/17 06:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/04/17 06:58:17 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.


Spark Version: 3.4.2

LOADING GOLD-LAYER DATA


26/04/17 06:58:19 WARN DFSPropertiesConfiguration: Cannot find HUDI_CONF_DIR, please set it as the dir of hudi-defaults.conf
26/04/17 06:58:19 WARN DFSPropertiesConfiguration: Properties file file:/etc/hudi/conf/hudi-defaults.conf not found. Ignoring to load props file


✓ Alerts loaded: 128378 records
✓ Cases loaded: 128381 records
✓ Tasks loaded: 128378 records
✓ Transactions loaded: 307786 records


## 2. Date Range & Granularity Configuration

In [3]:
def get_period_dates(period_name):
    """Get start and end dates for pre-configured periods"""
    end_date = datetime.now().date()
    
    if period_name == "Last 7 days":
        start_date = end_date - timedelta(days=7)
    elif period_name == "Last 30 days":
        start_date = end_date - timedelta(days=30)
    elif period_name == "Last 90 days":
        start_date = end_date - timedelta(days=90)
    elif period_name == "Last year":
        start_date = end_date - timedelta(days=365)
    elif period_name == "All history":
        start_date = datetime(2000, 1, 1).date()
    else:
        start_date = end_date - timedelta(days=90)
    
    return start_date, end_date

def get_previous_period(start_date, end_date):
    """Calculate equivalent previous period for trend comparison"""
    period_length = (end_date - start_date).days
    prev_end = start_date - timedelta(days=1)
    prev_start = prev_end - timedelta(days=period_length)
    return prev_start, prev_end

def get_trend_indicator(current, previous, threshold=5):
    """Generate trend indicator: ↑ (positive), ↓ (negative), → (neutral)"""
    if previous == 0 or pd.isna(current) or pd.isna(previous):
        return "→", "gray"
    
    pct_change = ((current - previous) / previous) * 100
    
    if pct_change > threshold:
        return "↑", "green"
    elif pct_change < -threshold:
        return "↓", "red"
    else:
        return "→", "gray"

# Configure dashboard parameters
selected_period = "Last 90 days"  # Options: "Last 7 days", "Last 30 days", "Last 90 days", "Last year", "All history"
selected_granularity = "Weekly"   # Options: "Daily", "Weekly", "Monthly", "Quarterly"

start_date, end_date = get_period_dates(selected_period)
prev_start_date, prev_end_date = get_previous_period(start_date, end_date)

print("\n" + "="*80)
print("DASHBOARD CONFIGURATION")
print("="*80)
print(f"Selected Period: {selected_period}")
print(f"Analysis Period: {start_date} to {end_date}")
print(f"Previous Period: {prev_start_date} to {prev_end_date}")
print(f"Time Granularity: {selected_granularity}")
print("="*80)


DASHBOARD CONFIGURATION
Selected Period: Last 90 days
Analysis Period: 2026-01-17 to 2026-04-17
Previous Period: 2025-10-18 to 2026-01-16
Time Granularity: Weekly


## 3. Alert Metrics Trends

In [4]:
def calculate_alert_metrics(start_date, end_date, prev_start_date, prev_end_date):
    """Calculate alert metrics trends: evaluated rate, received rate, auto-closure rate"""
    
    # Current period
    alerts_current = alerts.filter(
        (F.col("created_at_ts").cast("date") >= start_date) &
        (F.col("created_at_ts").cast("date") <= end_date)
    )
    
    cases_current = cases.filter(
        (F.col("case_created_ts").cast("date") >= start_date) &
        (F.col("case_created_ts").cast("date") <= end_date)
    )
    
    transactions_current = transactions.filter(
        (F.col("ingested_at_ts").cast("date") >= start_date) &
        (F.col("ingested_at_ts").cast("date") <= end_date)
    )
    
    # Previous period
    alerts_prev = alerts.filter(
        (F.col("created_at_ts").cast("date") >= prev_start_date) &
        (F.col("created_at_ts").cast("date") <= prev_end_date)
    )
    
    cases_prev = cases.filter(
        (F.col("case_created_ts").cast("date") >= prev_start_date) &
        (F.col("case_created_ts").cast("date") <= prev_end_date)
    )
    
    transactions_prev = transactions.filter(
        (F.col("ingested_at_ts").cast("date") >= prev_start_date) &
        (F.col("ingested_at_ts").cast("date") <= prev_end_date)
    )
    
    # 1a. Evaluated Alert Rate = transactions with alerts / total transactions evaluated
    tx_with_alerts_current = alerts_current.select("tx_msg_id").distinct().count()
    tx_evaluated_current = transactions_current.count()
    evaluated_alert_rate_current = (tx_with_alerts_current / tx_evaluated_current * 100) if tx_evaluated_current > 0 else 0
    
    tx_with_alerts_prev = alerts_prev.select("tx_msg_id").distinct().count()
    tx_evaluated_prev = transactions_prev.count()
    evaluated_alert_rate_prev = (tx_with_alerts_prev / tx_evaluated_prev * 100) if tx_evaluated_prev > 0 else 0
    
    # 1b. Received Alert Rate = transactions with alerts / total transactions received
    received_alert_rate_current = evaluated_alert_rate_current
    received_alert_rate_prev = evaluated_alert_rate_prev
    
    # 1c. Auto-Closure Rate = auto-closed alerts / total alerts
    auto_closed_current = alerts_current.join(
        cases_current.select("case_id"),
        "case_id", "inner"
    ).filter(
        (F.col("prediction_outcome_norm") == "FALSE_POSITIVE") |
        ((F.col("alert_type_norm") == "FRAUD") & 
         (F.col("prediction_outcome_norm") == "TRUE_POSITIVE"))
    ).select("alert_id").distinct().count()
    
    total_alerts_current = alerts_current.count()
    auto_closure_rate_current = (auto_closed_current / total_alerts_current * 100) if total_alerts_current > 0 else 0
    
    auto_closed_prev = alerts_prev.join(
        cases_prev.select("case_id"),
        "case_id", "inner"
    ).filter(
        (F.col("prediction_outcome_norm") == "FALSE_POSITIVE") |
        ((F.col("alert_type_norm") == "FRAUD") & 
         (F.col("prediction_outcome_norm") == "TRUE_POSITIVE"))
    ).select("alert_id").distinct().count()
    
    total_alerts_prev = alerts_prev.count()
    auto_closure_rate_prev = (auto_closed_prev / total_alerts_prev * 100) if total_alerts_prev > 0 else 0
    
    return {
        "evaluated_alert_rate_current": evaluated_alert_rate_current,
        "evaluated_alert_rate_prev": evaluated_alert_rate_prev,
        "received_alert_rate_current": received_alert_rate_current,
        "received_alert_rate_prev": received_alert_rate_prev,
        "auto_closure_rate_current": auto_closure_rate_current,
        "auto_closure_rate_prev": auto_closure_rate_prev,
        "tx_with_alerts_current": tx_with_alerts_current,
        "tx_evaluated_current": tx_evaluated_current,
        "total_alerts_current": total_alerts_current,
    }

# Calculate alert metrics
alert_metrics = calculate_alert_metrics(start_date, end_date, prev_start_date, prev_end_date)

print("\n" + "="*80)
print("ALERT METRICS")
print("="*80)

print(f"\n1a. Evaluated Alert Rate")
print(f"    Current: {alert_metrics['evaluated_alert_rate_current']:.2f}%")
print(f"    Previous: {alert_metrics['evaluated_alert_rate_prev']:.2f}%")
trend, color = get_trend_indicator(alert_metrics['evaluated_alert_rate_current'], 
                                    alert_metrics['evaluated_alert_rate_prev'])
print(f"    Trend: {trend}")

print(f"\n1b. Received Alert Rate")
print(f"    Current: {alert_metrics['received_alert_rate_current']:.2f}%")
print(f"    Previous: {alert_metrics['received_alert_rate_prev']:.2f}%")

print(f"\n1c. Auto-Closure Rate")
print(f"    Current: {alert_metrics['auto_closure_rate_current']:.2f}%")
print(f"    Previous: {alert_metrics['auto_closure_rate_prev']:.2f}%")
trend, color = get_trend_indicator(alert_metrics['auto_closure_rate_current'], 
                                    alert_metrics['auto_closure_rate_prev'], threshold=3)
print(f"    Trend: {trend}")

print("="*80)


ALERT METRICS

1a. Evaluated Alert Rate
    Current: 41.71%
    Previous: 0.00%
    Trend: →

1b. Received Alert Rate
    Current: 41.71%
    Previous: 0.00%

1c. Auto-Closure Rate
    Current: 25.00%
    Previous: 0.00%
    Trend: →


## 4. Case Inventory Metrics

In [5]:
def calculate_case_inventory_metrics(start_date, end_date, prev_start_date, prev_end_date):
    """Calculate case inflow, outflow, and reopened rates"""
    
    # Current period
    cases_current = cases.filter(
        (F.col("case_created_ts").cast("date") >= start_date) &
        (F.col("case_created_ts").cast("date") <= end_date)
    )
    
    alerts_current = alerts.filter(
        (F.col("created_at_ts").cast("date") >= start_date) &
        (F.col("created_at_ts").cast("date") <= end_date)
    )
    
    # Previous period
    cases_prev = cases.filter(
        (F.col("case_created_ts").cast("date") >= prev_start_date) &
        (F.col("case_created_ts").cast("date") <= prev_end_date)
    )
    
    alerts_prev = alerts.filter(
        (F.col("created_at_ts").cast("date") >= prev_start_date) &
        (F.col("created_at_ts").cast("date") <= prev_end_date)
    )
    
    # NEW CASES (INFLOW)
    # 2a-i. Cases from alerts
    cases_from_alerts_current = cases_current.filter(
        F.col("case_creation_type") == "AUTOMATIC_SYSTEM"
    ).count()
    
    cases_from_alerts_prev = cases_prev.filter(
        F.col("case_creation_type") == "AUTOMATIC_SYSTEM"
    ).count()
    
    # 2a-ii. Manually created cases
    manual_cases_current = cases_current.filter(
        F.col("case_creation_type") != "AUTOMATIC_SYSTEM"
    ).count()
    
    manual_cases_prev = cases_prev.filter(
        F.col("case_creation_type") != "AUTOMATIC_SYSTEM"
    ).count()
    
    # 2a-iii. AML cases
    aml_cases_current = cases_current.join(
        alerts_current.filter(
            (F.col("alert_type_norm") == "AML") | 
            (F.col("alert_type_norm") == "FRAUD_AND_AML")
        ).select("case_id"),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    aml_cases_prev = cases_prev.join(
        alerts_prev.filter(
            (F.col("alert_type_norm") == "AML") | 
            (F.col("alert_type_norm") == "FRAUD_AND_AML")
        ).select("case_id"),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    # 2a-iv. Fraud cases
    fraud_cases_current = cases_current.join(
        alerts_current.filter(
            (F.col("alert_type_norm") == "FRAUD") | 
            (F.col("alert_type_norm") == "FRAUD_AND_AML")
        ).select("case_id"),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    fraud_cases_prev = cases_prev.join(
        alerts_prev.filter(
            (F.col("alert_type_norm") == "FRAUD") | 
            (F.col("alert_type_norm") == "FRAUD_AND_AML")
        ).select("case_id"),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    # 2a-v. AML & Fraud cases
    fraud_aml_cases_current = cases_current.join(
        alerts_current.filter(
            F.col("alert_type_norm") == "FRAUD_AND_AML"
        ).select("case_id"),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    fraud_aml_cases_prev = cases_prev.join(
        alerts_prev.filter(
            F.col("alert_type_norm") == "FRAUD_AND_AML"
        ).select("case_id"),
        "case_id", "inner"
    ).select("case_id").distinct().count()
    
    # CLOSED CASES (OUTFLOW)
    # 2b-i. Auto-closed cases
    auto_closed_current = cases_current.filter(
        F.col("status").like("%AUTO%CLOSED%")
    ).count()
    
    auto_closed_prev = cases_prev.filter(
        F.col("status").like("%AUTO%CLOSED%")
    ).count()
    
    # 2b-ii. Total closed cases
    closed_cases_current = cases_current.filter(
        (F.col("status").like("%CLOSED%")) | 
        (F.col("status").like("%COMPLETE%")) |
        (F.col("status").like("%DISPOSED%"))
    ).count()
    
    closed_cases_prev = cases_prev.filter(
        (F.col("status").like("%CLOSED%")) | 
        (F.col("status").like("%COMPLETE%")) |
        (F.col("status").like("%DISPOSED%"))
    ).count()
    
    # 2c. Case Closure Rate per investigator (for current period)
    cases_per_investigator_current = cases_current.filter(
        F.col("case_owner_user_id").isNotNull()
    ).groupBy("case_owner_user_id").agg(
        F.count("case_id").alias("total_cases"),
        F.sum(F.when(
            (F.col("status").like("%CLOSED%")) | 
            (F.col("status").like("%COMPLETE%")) |
            (F.col("status").like("%DISPOSED%")), 1
        ).otherwise(0)).alias("closed_cases")
    ).withColumn(
        "closure_rate",
        F.round((F.col("closed_cases") / F.col("total_cases")) * 100, 2)
    )
    
    avg_closure_rate_current = cases_per_investigator_current.agg(
        F.avg("closure_rate")
    ).collect()[0][0] or 0
    
    cases_per_investigator_prev = cases_prev.filter(
        F.col("case_owner_user_id").isNotNull()
    ).groupBy("case_owner_user_id").agg(
        F.count("case_id").alias("total_cases"),
        F.sum(F.when(
            (F.col("status").like("%CLOSED%")) | 
            (F.col("status").like("%COMPLETE%")) |
            (F.col("status").like("%DISPOSED%")), 1
        ).otherwise(0)).alias("closed_cases")
    ).withColumn(
        "closure_rate",
        F.round((F.col("closed_cases") / F.col("total_cases")) * 100, 2)
    )
    
    avg_closure_rate_prev = cases_per_investigator_prev.agg(
        F.avg("closure_rate")
    ).collect()[0][0] or 0
    
    # REOPENED CASES
    # 2c. Reopened Case Rate = cases reopened / total closed cases
    reopened_cases_current = cases_current.filter(
        F.col("has_parent_case") == 1
    ).count()
    
    reopened_rate_current = (reopened_cases_current / closed_cases_current * 100) if closed_cases_current > 0 else 0
    
    reopened_cases_prev = cases_prev.filter(
        F.col("has_parent_case") == 1
    ).count()
    
    reopened_rate_prev = (reopened_cases_prev / closed_cases_prev * 100) if closed_cases_prev > 0 else 0
    
    return {
        "cases_from_alerts_current": cases_from_alerts_current,
        "cases_from_alerts_prev": cases_from_alerts_prev,
        "manual_cases_current": manual_cases_current,
        "manual_cases_prev": manual_cases_prev,
        "aml_cases_current": aml_cases_current,
        "aml_cases_prev": aml_cases_prev,
        "fraud_cases_current": fraud_cases_current,
        "fraud_cases_prev": fraud_cases_prev,
        "fraud_aml_cases_current": fraud_aml_cases_current,
        "fraud_aml_cases_prev": fraud_aml_cases_prev,
        "auto_closed_current": auto_closed_current,
        "auto_closed_prev": auto_closed_prev,
        "closed_cases_current": closed_cases_current,
        "closed_cases_prev": closed_cases_prev,
        "avg_closure_rate_current": avg_closure_rate_current,
        "avg_closure_rate_prev": avg_closure_rate_prev,
        "reopened_rate_current": reopened_rate_current,
        "reopened_rate_prev": reopened_rate_prev,
    }

# Calculate case inventory metrics
case_inventory = calculate_case_inventory_metrics(start_date, end_date, prev_start_date, prev_end_date)

print("\n" + "="*80)
print("CASE INVENTORY METRICS")
print("="*80)

print(f"\n2a. NEW CASES (INFLOW)")
print(f"    From Alerts: {case_inventory['cases_from_alerts_current']} (prev: {case_inventory['cases_from_alerts_prev']})")
print(f"    Manual Cases: {case_inventory['manual_cases_current']} (prev: {case_inventory['manual_cases_prev']})")
print(f"    AML Cases: {case_inventory['aml_cases_current']} (prev: {case_inventory['aml_cases_prev']})")
print(f"    Fraud Cases: {case_inventory['fraud_cases_current']} (prev: {case_inventory['fraud_cases_prev']})")
print(f"    AML & Fraud Cases: {case_inventory['fraud_aml_cases_current']} (prev: {case_inventory['fraud_aml_cases_prev']})")

print(f"\n2b. CLOSED CASES (OUTFLOW)")
print(f"    Auto-Closed: {case_inventory['auto_closed_current']} (prev: {case_inventory['auto_closed_prev']})")
print(f"    Total Closed: {case_inventory['closed_cases_current']} (prev: {case_inventory['closed_cases_prev']})")
print(f"    Avg Closure Rate: {case_inventory['avg_closure_rate_current']:.2f}% (prev: {case_inventory['avg_closure_rate_prev']:.2f}%)")

print(f"\n2c. REOPENED CASE RATE")
print(f"    Current: {case_inventory['reopened_rate_current']:.2f}%")
print(f"    Previous: {case_inventory['reopened_rate_prev']:.2f}%")
trend, color = get_trend_indicator(case_inventory['reopened_rate_current'], 
                                    case_inventory['reopened_rate_prev'], threshold=2)
print(f"    Trend: {trend} (lower is better)")

print("="*80)


CASE INVENTORY METRICS

2a. NEW CASES (INFLOW)
    From Alerts: 128381 (prev: 0)
    Manual Cases: 0 (prev: 0)
    AML Cases: 0 (prev: 0)
    Fraud Cases: 0 (prev: 0)
    AML & Fraud Cases: 0 (prev: 0)

2b. CLOSED CASES (OUTFLOW)
    Auto-Closed: 0 (prev: 0)
    Total Closed: 0 (prev: 0)
    Avg Closure Rate: 0.00% (prev: 0.00%)

2c. REOPENED CASE RATE
    Current: 0.00%
    Previous: 0.00%
    Trend: → (lower is better)


## 5. Resolution Time Metrics

In [6]:
def calculate_resolution_time_metrics(start_date, end_date, prev_start_date, prev_end_date):
    """Calculate MTTD, MTTD by disposition, and detection lag"""
    
    # Current period
    cases_current = cases.filter(
        (F.col("case_created_ts").cast("date") >= start_date) &
        (F.col("case_created_ts").cast("date") <= end_date) &
        (F.col("created_to_updated_ms") > 0)
    )
    
    # Previous period
    cases_prev = cases.filter(
        (F.col("case_created_ts").cast("date") >= prev_start_date) &
        (F.col("case_created_ts").cast("date") <= prev_end_date) &
        (F.col("created_to_updated_ms") > 0)
    )
    
    # 3a. Mean Time to Disposition (MTTD)
    mttd_ms_current = cases_current.agg(
        F.avg("created_to_updated_ms")
    ).collect()[0][0] or 0
    mttd_hours_current = mttd_ms_current / (1000 * 60 * 60)
    mttd_days_current = mttd_hours_current / 24
    
    mttd_ms_prev = cases_prev.agg(
        F.avg("created_to_updated_ms")
    ).collect()[0][0] or 0
    mttd_hours_prev = mttd_ms_prev / (1000 * 60 * 60)
    mttd_days_prev = mttd_hours_prev / 24
    
    # 3b. MTTD by disposition
    mttd_by_disposition_current = cases_current.groupBy("status").agg(
        F.avg("created_to_updated_ms").alias("avg_duration_ms"),
        F.count("case_id").alias("case_count")
    ).withColumn(
        "avg_duration_hours",
        F.round(F.col("avg_duration_ms") / (1000 * 60 * 60), 2)
    ).withColumn(
        "avg_duration_days",
        F.round(F.col("avg_duration_hours") / 24, 2)
    )
    
    # 3c. Fraud Detection Lag (time from transaction to confirmed fraud case)
    detection_lag_df = cases_current.join(
        alerts.select("case_id", "tx_msg_id"),
        "case_id", "inner"
    ).join(
        transactions.select("tx_msg_id", "event_ts"),
        "tx_msg_id", "left"
    ).withColumn(
        "detection_lag_ms",
        (F.unix_timestamp("case_created_ts") - F.unix_timestamp("event_ts")) * 1000
    ).filter(F.col("detection_lag_ms") > 0)
    
    detection_lag_hours_current = detection_lag_df.agg(
        F.avg("detection_lag_ms") / (1000 * 60 * 60)
    ).collect()[0][0] or 0
    
    detection_lag_df_prev = cases_prev.join(
        alerts.select("case_id", "tx_msg_id"),
        "case_id", "inner"
    ).join(
        transactions.select("tx_msg_id", "event_ts"),
        "tx_msg_id", "left"
    ).withColumn(
        "detection_lag_ms",
        (F.unix_timestamp("case_created_ts") - F.unix_timestamp("event_ts")) * 1000
    ).filter(F.col("detection_lag_ms") > 0)
    
    detection_lag_hours_prev = detection_lag_df_prev.agg(
        F.avg("detection_lag_ms") / (1000 * 60 * 60)
    ).collect()[0][0] or 0
    
    return {
        "mttd_hours_current": mttd_hours_current,
        "mttd_days_current": mttd_days_current,
        "mttd_hours_prev": mttd_hours_prev,
        "mttd_days_prev": mttd_days_prev,
        "mttd_by_disposition_current": mttd_by_disposition_current,
        "detection_lag_hours_current": detection_lag_hours_current,
        "detection_lag_hours_prev": detection_lag_hours_prev,
    }

# Calculate resolution time metrics
resolution_metrics = calculate_resolution_time_metrics(start_date, end_date, prev_start_date, prev_end_date)

print("\n" + "="*80)
print("RESOLUTION TIME METRICS")
print("="*80)

print(f"\n3a. MEAN TIME TO DISPOSITION (MTTD)")
print(f"    Current: {resolution_metrics['mttd_hours_current']:.2f} hours ({resolution_metrics['mttd_days_current']:.2f} days)")
print(f"    Previous: {resolution_metrics['mttd_hours_prev']:.2f} hours ({resolution_metrics['mttd_days_prev']:.2f} days)")
trend, color = get_trend_indicator(resolution_metrics['mttd_hours_current'], 
                                    resolution_metrics['mttd_hours_prev'], threshold=5)
print(f"    Trend: {trend} (lower is better)")

print(f"\n3b. MTTD BY DISPOSITION")
mttd_disp_df = resolution_metrics['mttd_by_disposition_current'].toPandas()
for idx, row in mttd_disp_df.iterrows():
    print(f"    {row['status']}: {row['avg_duration_hours']:.2f} hrs ({row['avg_duration_days']:.2f} days) - {int(row['case_count'])} cases")

print(f"\n3c. FRAUD DETECTION LAG")
print(f"    Current: {resolution_metrics['detection_lag_hours_current']:.2f} hours")
print(f"    Previous: {resolution_metrics['detection_lag_hours_prev']:.2f} hours")
trend, color = get_trend_indicator(resolution_metrics['detection_lag_hours_current'], 
                                    resolution_metrics['detection_lag_hours_prev'], threshold=5)
print(f"    Trend: {trend} (lower is better)")

print("="*80)


RESOLUTION TIME METRICS

3a. MEAN TIME TO DISPOSITION (MTTD)
    Current: 0.00 hours (0.00 days)
    Previous: 0.00 hours (0.00 days)
    Trend: → (lower is better)

3b. MTTD BY DISPOSITION

3c. FRAUD DETECTION LAG
    Current: 0.00 hours
    Previous: 0.00 hours
    Trend: → (lower is better)


## 6. Case Status & Investigation Metrics

In [7]:
def calculate_investigation_metrics(start_date, end_date, prev_start_date, prev_end_date):
    """Calculate investigation and task metrics"""
    
    # Current period
    cases_current = cases.filter(
        (F.col("case_created_ts").cast("date") >= start_date) &
        (F.col("case_created_ts").cast("date") <= end_date)
    )
    
    tasks_current = tasks.filter(
        (F.col("task_created_ts").cast("date") >= start_date) &
        (F.col("task_created_ts").cast("date") <= end_date)
    )
    
    # Previous period
    cases_prev = cases.filter(
        (F.col("case_created_ts").cast("date") >= prev_start_date) &
        (F.col("case_created_ts").cast("date") <= prev_end_date)
    )
    
    tasks_prev = tasks.filter(
        (F.col("task_created_ts").cast("date") >= prev_start_date) &
        (F.col("task_created_ts").cast("date") <= prev_end_date)
    )
    
    # Case Status Distribution
    cases_by_status_current = cases.groupBy("status").agg(
        F.count("case_id").alias("case_count")
    ).withColumn(
        "percentage",
        F.round((F.col("case_count") / F.sum("case_count").over(Window.partitionBy())) * 100, 2)
    ).orderBy(F.desc("case_count"))
    
    # 5. Open Tasks
    open_tasks_current = tasks_current.filter(F.col("is_completed") == 0).count()
    open_tasks_prev = tasks_prev.filter(F.col("is_completed") == 0).count()
    
    # Tasks by status
    tasks_by_status = tasks_current.groupBy("status").agg(
        F.count("task_id").alias("task_count")
    ).orderBy(F.desc("task_count"))
    
    # Open tasks by type
    open_tasks_by_type = tasks_current.filter(
        F.col("is_completed") == 0
    ).groupBy("candidate_group").agg(
        F.count("task_id").alias("task_count")
    ).orderBy(F.desc("task_count"))
    
    # Investigation Metrics
    # Investigations created = tasks with status "INVESTIGATION"
    investigations_created_current = tasks_current.filter(
        F.col("candidate_group").like("%INVEST%")
    ).select("case_id").distinct().count()
    
    investigations_created_prev = tasks_prev.filter(
        F.col("candidate_group").like("%INVEST%")
    ).select("case_id").distinct().count()
    
    # Open investigations
    open_investigations_current = tasks_current.filter(
        (F.col("candidate_group").like("%INVEST%")) &
        (F.col("is_completed") == 0)
    ).select("case_id").distinct().count()
    
    # Cases from alerts (for conversion rate calculation)
    cases_from_alerts_current = cases_current.filter(
        F.col("case_creation_type") == "AUTOMATIC_SYSTEM"
    ).count()
    
    cases_from_alerts_prev = cases_prev.filter(
        F.col("case_creation_type") == "AUTOMATIC_SYSTEM"
    ).count()
    
    # Investigation Conversion Rate
    investigation_conversion_rate_current = (open_investigations_current / cases_from_alerts_current * 100) \
        if cases_from_alerts_current > 0 else 0
    
    investigation_conversion_rate_prev = 0  # Would need previous period calculation
    
    # Confirmation Rate
    confirmed_cases_current = cases_current.filter(
        F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
    ).count()
    
    closed_cases_current = cases_current.filter(
        (F.col("status").like("%CLOSED%")) | 
        (F.col("status").like("%COMPLETE%")) |
        (F.col("status").like("%DISPOSED%"))
    ).count()
    
    confirmation_rate_current = (confirmed_cases_current / closed_cases_current * 100) \
        if closed_cases_current > 0 else 0
    
    confirmed_cases_prev = cases_prev.filter(
        F.col("status") == "STATUS_82_CLOSED_CONFIRMED"
    ).count()
    
    closed_cases_prev = cases_prev.filter(
        (F.col("status").like("%CLOSED%")) | 
        (F.col("status").like("%COMPLETE%")) |
        (F.col("status").like("%DISPOSED%"))
    ).count()
    
    confirmation_rate_prev = (confirmed_cases_prev / closed_cases_prev * 100) \
        if closed_cases_prev > 0 else 0
    
    return {
        "cases_by_status": cases_by_status_current,
        "open_tasks_current": open_tasks_current,
        "open_tasks_prev": open_tasks_prev,
        "tasks_by_status": tasks_by_status,
        "open_tasks_by_type": open_tasks_by_type,
        "investigations_created_current": investigations_created_current,
        "investigations_created_prev": investigations_created_prev,
        "open_investigations_current": open_investigations_current,
        "investigation_conversion_rate_current": investigation_conversion_rate_current,
        "confirmation_rate_current": confirmation_rate_current,
        "confirmation_rate_prev": confirmation_rate_prev,
    }

# Calculate investigation metrics
investigation_metrics = calculate_investigation_metrics(start_date, end_date, prev_start_date, prev_end_date)

print("\n" + "="*80)
print("CASE STATUS & INVESTIGATION METRICS")
print("="*80)

print(f"\n4. CASE STATUS DISTRIBUTION")
status_df = investigation_metrics['cases_by_status'].toPandas()
for idx, row in status_df.iterrows():
    print(f"    {row['status']}: {int(row['case_count'])} cases ({row['percentage']:.1f}%)")

print(f"\n5. OPEN TASKS")
print(f"    Current: {investigation_metrics['open_tasks_current']} tasks")
print(f"    Previous: {investigation_metrics['open_tasks_prev']} tasks")

print(f"\n5a. TASKS BY STATUS")
tasks_status_df = investigation_metrics['tasks_by_status'].toPandas()
for idx, row in tasks_status_df.iterrows():
    print(f"    {row['status']}: {int(row['task_count'])} tasks")

print(f"\n5b. OPEN TASKS BY TYPE")
tasks_type_df = investigation_metrics['open_tasks_by_type'].toPandas()
for idx, row in tasks_type_df.iterrows():
    print(f"    {row['candidate_group']}: {int(row['task_count'])} tasks")

print(f"\n6. INVESTIGATION METRICS")
print(f"    Investigations Created (Current): {investigation_metrics['investigations_created_current']}")
print(f"    Investigations Created (Previous): {investigation_metrics['investigations_created_prev']}")
print(f"    Open Investigations: {investigation_metrics['open_investigations_current']}")

print(f"\n6e. INVESTIGATION CONVERSION RATE")
print(f"    Current: {investigation_metrics['investigation_conversion_rate_current']:.2f}%")

print(f"\n6f. CONFIRMATION RATE")
print(f"    Current: {investigation_metrics['confirmation_rate_current']:.2f}%")
print(f"    Previous: {investigation_metrics['confirmation_rate_prev']:.2f}%")
trend, color = get_trend_indicator(investigation_metrics['confirmation_rate_current'], 
                                    investigation_metrics['confirmation_rate_prev'])
print(f"    Trend: {trend}")

print("="*80)


CASE STATUS & INVESTIGATION METRICS

4. CASE STATUS DISTRIBUTION


26/04/17 06:58:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:58:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:58:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:58:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:58:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/17 06:58:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


    STATUS_00_DRAFT: 128381 cases (100.0%)

5. OPEN TASKS
    Current: 128378 tasks
    Previous: 0 tasks

5a. TASKS BY STATUS
    STATUS_01_UNASSIGNED: 128378 tasks

5b. OPEN TASKS BY TYPE
    INVESTIGATIONS: 128378 tasks

6. INVESTIGATION METRICS
    Investigations Created (Current): 128378
    Investigations Created (Previous): 0
    Open Investigations: 128378

6e. INVESTIGATION CONVERSION RATE
    Current: 100.00%

6f. CONFIRMATION RATE
    Current: 0.00%
    Previous: 0.00%
    Trend: →


## 7. Workload & Investigator Productivity

In [8]:
def calculate_investigator_productivity(start_date, end_date):
    """Calculate workload distribution and investigator productivity metrics"""
    
    cases_period = cases.filter(
        (F.col("case_created_ts").cast("date") >= start_date) &
        (F.col("case_created_ts").cast("date") <= end_date)
    )
    
    # Workload per investigator
    investigator_workload = cases_period.filter(
        F.col("case_owner_user_id").isNotNull()
    ).groupBy("case_owner_user_id").agg(
        F.count("case_id").alias("total_cases"),
        F.sum(F.when(
            (F.col("status").like("%CLOSED%")) | 
            (F.col("status").like("%COMPLETE%")) |
            (F.col("status").like("%DISPOSED%")), 1
        ).otherwise(0)).alias("closed_cases"),
        F.sum(F.when(
            F.col("status") == "STATUS_82_CLOSED_CONFIRMED", 1
        ).otherwise(0)).alias("confirmed_cases"),
        F.avg("created_to_updated_ms").alias("avg_resolution_time_ms")
    ).withColumn(
        "closure_rate",
        F.round((F.col("closed_cases") / F.col("total_cases")) * 100, 2)
    ).withColumn(
        "confirmation_rate",
        F.round((F.col("confirmed_cases") / F.col("closed_cases")) * 100, 2)
    ).withColumn(
        "avg_resolution_hours",
        F.round(F.col("avg_resolution_time_ms") / (1000 * 60 * 60), 2)
    ).orderBy(F.desc("total_cases"))
    
    # Workload statistics
    total_investigators = investigator_workload.count()
    avg_cases_per_investigator = cases_period.filter(
        F.col("case_owner_user_id").isNotNull()
    ).select("case_owner_user_id").distinct().count()
    
    # Get top investigators
    top_investigators = investigator_workload.limit(5)
    
    # Workload distribution (volume buckets)
    workload_distribution = investigator_workload.groupBy(
        F.when(F.col("total_cases") >= 20, "High (20+)")
         .when(F.col("total_cases") >= 10, "Medium (10-19)")
         .when(F.col("total_cases") >= 5, "Low (5-9)")
         .otherwise("Very Low (<5)").alias("workload_level")
    ).agg(
        F.count("case_owner_user_id").alias("investigator_count")
    ).orderBy("workload_level")
    
    return {
        "investigator_workload": investigator_workload,
        "total_investigators": total_investigators,
        "avg_cases_per_investigator": avg_cases_per_investigator,
        "top_investigators": top_investigators,
        "workload_distribution": workload_distribution,
    }

# Calculate investigator productivity
productivity_metrics = calculate_investigator_productivity(start_date, end_date)

print("\n" + "="*80)
print("INVESTIGATOR PRODUCTIVITY & WORKLOAD METRICS")
print("="*80)

print(f"\nTotal Investigators: {productivity_metrics['total_investigators']}")
print(f"Active Investigators (with assigned cases): {productivity_metrics['avg_cases_per_investigator']}")

print(f"\nTOP 5 INVESTIGATORS BY WORKLOAD:")
top_inv_df = productivity_metrics['top_investigators'].toPandas()
for idx, row in top_inv_df.iterrows():
    print(f"  {int(idx+1)}. ID: {row['case_owner_user_id']}")
    print(f"     Total Cases: {int(row['total_cases'])}")
    print(f"     Closed Cases: {int(row['closed_cases'])} (Closure Rate: {row['closure_rate']:.1f}%)")
    print(f"     Confirmed Cases: {int(row['confirmed_cases'])} (Confirmation Rate: {row['confirmation_rate']:.1f}%)")
    print(f"     Avg Resolution Time: {row['avg_resolution_hours']:.1f} hours")

print(f"\nWORKLOAD DISTRIBUTION:")
workload_dist_df = productivity_metrics['workload_distribution'].toPandas()
for idx, row in workload_dist_df.iterrows():
    print(f"  {row['workload_level']}: {int(row['investigator_count'])} investigators")

print("="*80)


INVESTIGATOR PRODUCTIVITY & WORKLOAD METRICS

Total Investigators: 0
Active Investigators (with assigned cases): 0

TOP 5 INVESTIGATORS BY WORKLOAD:

WORKLOAD DISTRIBUTION:


## 8. Trend Visualizations

In [9]:
def create_kpi_card(value, previous_value, title, format_type="number", inverted=False):
    """Create KPI card with trend indicator using Plotly Indicator"""
    
    trend_symbol, trend_color = get_trend_indicator(value, previous_value, threshold=5)
    
    # Invert color logic for metrics where lower is better
    if inverted and trend_symbol == "↑":
        trend_color = "red"
    elif inverted and trend_symbol == "↓":
        trend_color = "green"
    
    if format_type == "percentage":
        display_value = f"{value:.2f}%"
        delta_value = f"{value - previous_value:.2f}%"
    elif format_type == "hours":
        display_value = f"{value:.1f}h"
        delta_value = f"{value - previous_value:.1f}h"
    elif format_type == "days":
        display_value = f"{value:.1f}d"
        delta_value = f"{value - previous_value:.1f}d"
    else:
        display_value = f"{value:.0f}"
        delta_value = f"{value - previous_value:.0f}"
    
    return go.Indicator(
        mode="number+delta",
        value=value,
        title={"text": title, "font": {"size": 14}},
        number={"valueformat": ".2f"},
        delta={
            "reference": previous_value,
            "valueformat": ".2f",
            "relative": False,
            "font": {"color": trend_color}
        },
        domain={"x": [0, 1], "y": [0, 1]}
    )

# Create comprehensive KPI dashboard with trend indicators
print("\n" + "="*80)
print("GENERATING KPI TREND VISUALIZATIONS")
print("="*80)

fig = make_subplots(
    rows=5, cols=3,
    subplot_titles=(
        "Evaluated Alert Rate (%)", "Received Alert Rate (%)", "Auto-Closure Rate (%)",
        "New Cases (From Alerts)", "Closed Cases", "Reopened Case Rate (%)",
        "MTTD (Hours)", "Detection Lag (Hours)", "Confirmation Rate (%)",
        "Open Tasks", "Open Investigations", "Investigation Conversion Rate (%)",
        "Avg Investigator Closure Rate (%)", "Avg Investigator Confirmation Rate (%)", "Avg Resolution Time (Hours)"
    ),
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Row 1: Alert Metrics
fig.add_trace(
    create_kpi_card(
        alert_metrics['evaluated_alert_rate_current'],
        alert_metrics['evaluated_alert_rate_prev'],
        "Evaluated Alert Rate"
    ),
    row=1, col=1
)

fig.add_trace(
    create_kpi_card(
        alert_metrics['received_alert_rate_current'],
        alert_metrics['received_alert_rate_prev'],
        "Received Alert Rate"
    ),
    row=1, col=2
)

fig.add_trace(
    create_kpi_card(
        alert_metrics['auto_closure_rate_current'],
        alert_metrics['auto_closure_rate_prev'],
        "Auto-Closure Rate"
    ),
    row=1, col=3
)

# Row 2: Case Inventory
fig.add_trace(
    create_kpi_card(
        case_inventory['cases_from_alerts_current'],
        case_inventory['cases_from_alerts_prev'],
        "New Cases (From Alerts)",
        format_type="number"
    ),
    row=2, col=1
)

fig.add_trace(
    create_kpi_card(
        case_inventory['closed_cases_current'],
        case_inventory['closed_cases_prev'],
        "Closed Cases",
        format_type="number"
    ),
    row=2, col=2
)

fig.add_trace(
    create_kpi_card(
        case_inventory['reopened_rate_current'],
        case_inventory['reopened_rate_prev'],
        "Reopened Case Rate",
        format_type="percentage",
        inverted=True
    ),
    row=2, col=3
)

# Row 3: Resolution Time
fig.add_trace(
    create_kpi_card(
        resolution_metrics['mttd_hours_current'],
        resolution_metrics['mttd_hours_prev'],
        "MTTD",
        format_type="hours",
        inverted=True
    ),
    row=3, col=1
)

fig.add_trace(
    create_kpi_card(
        resolution_metrics['detection_lag_hours_current'],
        resolution_metrics['detection_lag_hours_prev'],
        "Detection Lag",
        format_type="hours",
        inverted=True
    ),
    row=3, col=2
)

fig.add_trace(
    create_kpi_card(
        investigation_metrics['confirmation_rate_current'],
        investigation_metrics['confirmation_rate_prev'],
        "Confirmation Rate",
        format_type="percentage"
    ),
    row=3, col=3
)

# Row 4: Investigation Metrics
fig.add_trace(
    create_kpi_card(
        investigation_metrics['open_tasks_current'],
        investigation_metrics['open_tasks_prev'],
        "Open Tasks",
        format_type="number"
    ),
    row=4, col=1
)

fig.add_trace(
    create_kpi_card(
        investigation_metrics['open_investigations_current'],
        0,
        "Open Investigations",
        format_type="number"
    ),
    row=4, col=2
)

fig.add_trace(
    create_kpi_card(
        investigation_metrics['investigation_conversion_rate_current'],
        0,
        "Investigation Conversion Rate",
        format_type="percentage"
    ),
    row=4, col=3
)

# Row 5: Investigator Productivity
avg_closure_rate = productivity_metrics['investigator_workload'].agg(
    F.avg("closure_rate")
).collect()[0][0] or 0

avg_confirmation_rate = productivity_metrics['investigator_workload'].agg(
    F.avg("confirmation_rate")
).collect()[0][0] or 0

avg_resolution_time = productivity_metrics['investigator_workload'].agg(
    F.avg("avg_resolution_hours")
).collect()[0][0] or 0

fig.add_trace(
    create_kpi_card(
        avg_closure_rate,
        0,
        "Avg Closure Rate",
        format_type="percentage"
    ),
    row=5, col=1
)

fig.add_trace(
    create_kpi_card(
        avg_confirmation_rate,
        0,
        "Avg Confirmation Rate",
        format_type="percentage"
    ),
    row=5, col=2
)

fig.add_trace(
    create_kpi_card(
        avg_resolution_time,
        0,
        "Avg Resolution Time",
        format_type="hours",
        inverted=True
    ),
    row=5, col=3
)

fig.update_layout(
    title_text=f"Case Management Trend Dashboard - {selected_period}",
    title_font_size=24,
    height=1400,
    width=1600,
    showlegend=False
)

fig.show()

print("\n✓ KPI trend dashboard generated")
print("="*80)


GENERATING KPI TREND VISUALIZATIONS



✓ KPI trend dashboard generated


## 9. Supervisory Insights & Trends Analysis

In [10]:
print("\n" + "="*80)
print("SUPERVISORY INSIGHTS & TREND ANALYSIS")
print("="*80)

insights = []

# Alert Processing Insights
print("\n📊 ALERT PROCESSING EFFICIENCY:")
if alert_metrics['evaluated_alert_rate_current'] > 5:
    insights.append(f"⚠️  HIGH ALERT RATE: {alert_metrics['evaluated_alert_rate_current']:.2f}% of transactions generating alerts")
else:
    insights.append(f"✓ CONTROLLED ALERT RATE: {alert_metrics['evaluated_alert_rate_current']:.2f}% - within normal range")

if alert_metrics['auto_closure_rate_current'] > 40:
    insights.append(f"✓ HIGH AUTO-CLOSURE RATE: {alert_metrics['auto_closure_rate_current']:.2f}% of alerts automatically resolved")
else:
    insights.append(f"⚠️  LOW AUTO-CLOSURE RATE: {alert_metrics['auto_closure_rate_current']:.2f}% - more manual review needed")

for insight in insights:
    print(f"   {insight}")

# Case Inventory Insights
print("\n📋 CASE MANAGEMENT VOLUME:")
case_change = case_inventory['cases_from_alerts_current'] - case_inventory['cases_from_alerts_prev']
if case_change > 0:
    print(f"   📈 RISING WORKLOAD: +{case_change} new cases from alerts (trend: ↑)")
else:
    print(f"   📉 DECLINING WORKLOAD: {case_change} cases from alerts (trend: ↓)")

inflow_outflow = case_inventory['cases_from_alerts_current'] - case_inventory['closed_cases_current']
if inflow_outflow > 0:
    print(f"   ⚠️  CASE BACKLOG BUILDING: {inflow_outflow} more cases created than closed")
else:
    print(f"   ✓ BALANCED THROUGHPUT: Case closure tracking inflow")

# Resolution Time Insights
print("\n⏱️  CASE RESOLUTION EFFICIENCY:")
if resolution_metrics['mttd_hours_current'] > 48:
    print(f"   ⚠️  SLOW DISPOSITION: {resolution_metrics['mttd_days_current']:.1f} days average (above target)")
else:
    print(f"   ✓ EFFICIENT DISPOSITION: {resolution_metrics['mttd_days_current']:.1f} days average (on track)")

mttd_trend = "↑ SLOWING" if resolution_metrics['mttd_hours_current'] > resolution_metrics['mttd_hours_prev'] else "↓ IMPROVING"
print(f"   Trend: {mttd_trend}")

# Investigator Productivity Insights
print("\n👥 INVESTIGATOR PERFORMANCE:")
if productivity_metrics['total_investigators'] > 0:
    avg_workload = case_inventory['cases_from_alerts_current'] / productivity_metrics['total_investigators']
    print(f"   {productivity_metrics['total_investigators']} investigators")
    print(f"   Average workload: {avg_workload:.1f} cases per investigator")
    
    # Workload balance assessment
    workload_dist = productivity_metrics['workload_distribution'].toPandas()
    high_load = workload_dist[workload_dist['workload_level'] == 'High (20+)']['investigator_count'].sum()
    if high_load > productivity_metrics['total_investigators'] * 0.3:
        print(f"   ⚠️  UNBALANCED WORKLOAD: {int(high_load)} investigators overloaded (>20 cases)")
    else:
        print(f"   ✓ BALANCED WORKLOAD: Investigator case load well-distributed")

# Confirmation Rate Quality Insights
print("\n✅ CASE QUALITY METRICS:")
if investigation_metrics['confirmation_rate_current'] > 70:
    print(f"   ✓ HIGH CONFIRMATION RATE: {investigation_metrics['confirmation_rate_current']:.2f}% of closed cases confirmed as fraud/AML")
elif investigation_metrics['confirmation_rate_current'] < 40:
    print(f"   ⚠️  LOW CONFIRMATION RATE: {investigation_metrics['confirmation_rate_current']:.2f}% - high false positive rate likely")
else:
    print(f"   → MODERATE CONFIRMATION RATE: {investigation_metrics['confirmation_rate_current']:.2f}% - acceptable range")

# Task Management Insights
print("\n📌 TASK MANAGEMENT:")
open_task_change = investigation_metrics['open_tasks_current'] - investigation_metrics['open_tasks_prev']
if open_task_change > 0:
    print(f"   📈 TASK BACKLOG GROWING: +{open_task_change} open tasks (potential bottleneck)")
else:
    print(f"   ✓ TASK BACKLOG DECREASING: {open_task_change} tasks cleared (efficiency improving)")

print("\n" + "="*80)


SUPERVISORY INSIGHTS & TREND ANALYSIS

📊 ALERT PROCESSING EFFICIENCY:
   ⚠️  HIGH ALERT RATE: 41.71% of transactions generating alerts
   ⚠️  LOW AUTO-CLOSURE RATE: 25.00% - more manual review needed

📋 CASE MANAGEMENT VOLUME:
   📈 RISING WORKLOAD: +128381 new cases from alerts (trend: ↑)
   ⚠️  CASE BACKLOG BUILDING: 128381 more cases created than closed

⏱️  CASE RESOLUTION EFFICIENCY:
   ✓ EFFICIENT DISPOSITION: 0.0 days average (on track)
   Trend: ↓ IMPROVING

👥 INVESTIGATOR PERFORMANCE:

✅ CASE QUALITY METRICS:
   ⚠️  LOW CONFIRMATION RATE: 0.00% - high false positive rate likely

📌 TASK MANAGEMENT:
   📈 TASK BACKLOG GROWING: +128378 open tasks (potential bottleneck)



## 10. Summary Dashboard Table

In [11]:
def create_summary_dataframe(alert_metrics, case_inventory, resolution_metrics, investigation_metrics):
    """Create comprehensive summary dataframe with all metrics and trends"""
    
    summary_data = []
    
    # Alert Metrics
    summary_data.append({
        "Category": "Alert Metrics",
        "Metric": "Evaluated Alert Rate (%)",
        "Current": f"{alert_metrics['evaluated_alert_rate_current']:.2f}%",
        "Previous": f"{alert_metrics['evaluated_alert_rate_prev']:.2f}%",
        "Trend": get_trend_indicator(alert_metrics['evaluated_alert_rate_current'], 
                                      alert_metrics['evaluated_alert_rate_prev'])[0]
    })
    
    summary_data.append({
        "Category": "Alert Metrics",
        "Metric": "Auto-Closure Rate (%)",
        "Current": f"{alert_metrics['auto_closure_rate_current']:.2f}%",
        "Previous": f"{alert_metrics['auto_closure_rate_prev']:.2f}%",
        "Trend": get_trend_indicator(alert_metrics['auto_closure_rate_current'], 
                                      alert_metrics['auto_closure_rate_prev'])[0]
    })
    
    # Case Inventory
    summary_data.append({
        "Category": "Case Inventory",
        "Metric": "New Cases (From Alerts)",
        "Current": f"{int(case_inventory['cases_from_alerts_current'])}",
        "Previous": f"{int(case_inventory['cases_from_alerts_prev'])}",
        "Trend": get_trend_indicator(case_inventory['cases_from_alerts_current'], 
                                      case_inventory['cases_from_alerts_prev'])[0]
    })
    
    summary_data.append({
        "Category": "Case Inventory",
        "Metric": "Closed Cases",
        "Current": f"{int(case_inventory['closed_cases_current'])}",
        "Previous": f"{int(case_inventory['closed_cases_prev'])}",
        "Trend": get_trend_indicator(case_inventory['closed_cases_current'], 
                                      case_inventory['closed_cases_prev'])[0]
    })
    
    summary_data.append({
        "Category": "Case Inventory",
        "Metric": "Reopened Case Rate (%)",
        "Current": f"{case_inventory['reopened_rate_current']:.2f}%",
        "Previous": f"{case_inventory['reopened_rate_prev']:.2f}%",
        "Trend": get_trend_indicator(case_inventory['reopened_rate_prev'], 
                                      case_inventory['reopened_rate_current'])[0]  # Inverted
    })
    
    # Resolution Time
    summary_data.append({
        "Category": "Resolution Time",
        "Metric": "MTTD (Hours)",
        "Current": f"{resolution_metrics['mttd_hours_current']:.2f}",
        "Previous": f"{resolution_metrics['mttd_hours_prev']:.2f}",
        "Trend": get_trend_indicator(resolution_metrics['mttd_hours_prev'], 
                                      resolution_metrics['mttd_hours_current'])[0]  # Inverted
    })
    
    summary_data.append({
        "Category": "Resolution Time",
        "Metric": "Detection Lag (Hours)",
        "Current": f"{resolution_metrics['detection_lag_hours_current']:.2f}",
        "Previous": f"{resolution_metrics['detection_lag_hours_prev']:.2f}",
        "Trend": get_trend_indicator(resolution_metrics['detection_lag_hours_prev'], 
                                      resolution_metrics['detection_lag_hours_current'])[0]  # Inverted
    })
    
    # Investigation Metrics
    summary_data.append({
        "Category": "Investigation",
        "Metric": "Open Tasks",
        "Current": f"{int(investigation_metrics['open_tasks_current'])}",
        "Previous": f"{int(investigation_metrics['open_tasks_prev'])}",
        "Trend": get_trend_indicator(investigation_metrics['open_tasks_prev'], 
                                      investigation_metrics['open_tasks_current'])[0]  # Inverted
    })
    
    summary_data.append({
        "Category": "Investigation",
        "Metric": "Confirmation Rate (%)",
        "Current": f"{investigation_metrics['confirmation_rate_current']:.2f}%",
        "Previous": f"{investigation_metrics['confirmation_rate_prev']:.2f}%",
        "Trend": get_trend_indicator(investigation_metrics['confirmation_rate_current'], 
                                      investigation_metrics['confirmation_rate_prev'])[0]
    })
    
    summary_data.append({
        "Category": "Investigation",
        "Metric": "Investigation Conversion Rate (%)",
        "Current": f"{investigation_metrics['investigation_conversion_rate_current']:.2f}%",
        "Previous": "N/A",
        "Trend": "→"
    })
    
    return pd.DataFrame(summary_data)

# Create summary dataframe
summary_df = create_summary_dataframe(alert_metrics, case_inventory, resolution_metrics, investigation_metrics)

# Create interactive table visualization
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Category</b>", "<b>Metric</b>", "<b>Current</b>", "<b>Previous</b>", "<b>Trend</b>"],
        fill_color="#f0f2f5",
        align="left",
        font=dict(color="black", size=12, family="Arial"),
        height=30
    ),
    cells=dict(
        values=[summary_df['Category'], summary_df['Metric'], summary_df['Current'], 
                summary_df['Previous'], summary_df['Trend']],
        fill_color=[["#f9f9f9", "#ffffff"] * 5],
        align="left",
        font=dict(size=11),
        height=28
    )
)])

fig_table.update_layout(
    title="Case Management Metrics Summary",
    height=600,
    width=1200,
    margin=dict(l=0, r=0, t=40, b=0)
)

fig_table.show()

print("\n✓ Summary table generated")
print("="*80)


✓ Summary table generated
